In [ ]:
%load_ext autoreload
%autoreload 2

import polars as pl
import pandas as pd
from anngeno import AnnGeno
from tqdm import tqdm

from plotnine import *
import matplotlib.pyplot as plt

In [ ]:
anngeno_path = "/home/dnanexus/data_dir/genebass_1e6_coding_variants.ag"
ag = AnnGeno(anngeno_path,  filemode="r", low_mem=True)
ag

In [ ]:
maf = 0.001
variants_to_keep = ag.annotations.filter((pl.col('AF_ukb') < maf)).select('id').collect()['id']
ag.subset_variants(set(variants_to_keep))

In [ ]:
# gl = ag.annotations.select(pl.col("region")).unique().collect()['region'].to_list()

# Gene intersecion with maveDB
gl = ['ENSG00000092148',
 'ENSG00000181722',
 'ENSG00000101191',
 'ENSG00000012048',
 'ENSG00000029534',
 'ENSG00000134769',
 'ENSG00000078900',
 'ENSG00000136068',
 'ENSG00000138411',
 'ENSG00000139618',
 'ENSG00000035928',
 'ENSG00000101752',
 'ENSG00000025293',
 'ENSG00000165097',
 'ENSG00000105392',
 'ENSG00000141582',
 'ENSG00000175520',
 'ENSG00000163132',
 'ENSG00000180900',
 'ENSG00000163914',
 'ENSG00000170004',
 'ENSG00000170374',
 'ENSG00000145715',
 'ENSG00000183765',
 'ENSG00000135679',
 'ENSG00000189079',
 'ENSG00000106633',
 'ENSG00000108381',
 'ENSG00000163554',
 'ENSG00000117400',
 'ENSG00000005513',
 'ENSG00000089902',
 'ENSG00000166326']

In [ ]:
ag.samples

In [ ]:
%%time

result = []

rd = ag.get_many_regions(gl)

for gene_id in tqdm(rd.keys()):
    sums = rd[gene_id]['genotypes'].sum(axis=0)
    counts = pl.DataFrame({"sums": sums}).group_by("sums").len().rename({"len": "count"})
    counts = counts.with_columns(
        pl.lit(gene_id).alias("gene_id")
    )
    result.append(counts)

final_df = pl.concat(result)
final_df

In [ ]:
# Read genebass for gene symbols

gb_res = pd.read_parquet('/home/dnanexus/data_dir/genebass_continuous_associations_ukbbgym.pq')

pval_cutoffs = {'burden': 6.7e-7, 'skato': 2.5e-7} # pvalue thresholds used in genebass paper
mask = (gb_res['Pvalue'] < pval_cutoffs['skato'])
mask |= (gb_res['Pvalue_Burden'] < pval_cutoffs['burden'])
gb_res["significant"] = mask

gb_res = pl.from_pandas(gb_res.query("(significant == True) & (trait_type=='continuous') & (modifier != 'custom') & ('pLoF' in annotation)"))
gb_res

In [ ]:
final_df = final_df.join(gb_res.select(['gene_id', 'gene_symbol']).unique(), on='gene_id', how='left')
final_df

In [ ]:
(
    ggplot(final_df, aes(x='gene_symbol', y='count', fill='factor(sums)')) +  # treat sums as categorical
    geom_col(position='dodge') +
    scale_y_log10() +
    annotation_logticks(sides='l') +
    theme_bw() +
    theme(
        axis_text_x=element_text(rotation=90),
        figure_size=(14, 6),
    ) # `ha` for horizontal alignment
)

## Old code

In [ ]:
%%time

var_counts = []

rd = ag.get_many_regions(gl)

for gene_id in tqdm(rd.keys()):
    var_counts.extend(rd[gene_id]['genotypes'].sum(axis=0))

vdf = pl.DataFrame({
    "gene_id": [gene_id for gene_id in rd.keys() for _ in range(ag.samples.shape[0])],
    "sample_id": list(ag.samples) * len(rd.keys()),
    "var_counts": pl.Series(var_counts, dtype=pl.UInt16)
})

vdf

In [ ]:
(
    ggplot(vdf, aes(y='var_counts', x='gene_id')) +
    geom_boxplot() +
    theme_bw() +
    theme(axis_text_x=element_text(rotation=90, hjust=1))
)